<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Hyperparameter Tuning and Regularization</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/practice/week12/practice12_hyperparameter_tuning_and_regularization.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) / [Practice Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Practice_index.ipynb)

### ME 5995 - AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science
**Wayne State University / Optional, ungraded self-study / Normally not submitted**  
**Version: Student starter**

Try your selected exercises before consulting the solution. Each answer is one
possible approach. [Open the self-check solution](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/practice/week12/practice12_hyperparameter_tuning_and_regularization_solution.ipynb).

## Student Information

Optional for your personal copy. Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Part A. What changes when you change one setting?

Allow about 40 minutes for both parts. Run the cells in order. Part A uses NumPy and Matplotlib. Part B also uses pandas and TensorFlow. In Colab, open the notebook and run the first cell. For local work, install these packages in your Python environment.

A **hyperparameter** is a choice you make for a training run, such as learning rate or batch size. A **weight** is a value the network learns. **Regularization** helps reduce overfitting. Here you will try dropout and use validation loss to decide when to stop training.

The eight points below have two dimensionless features. Four fixed hidden units keep the positive parts of x1, -x1, x2 and -x2. Only the four weights leading to class B change. Class A always scores zero. Where the B score is also zero, the two classes tie.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Each row is one point. The labels use 1 for class B and 0 for class A.
# This fixed order lets a larger batch include the same first point.
points = np.array([[1.7, -0.6], [-1.0, 0.7], [-1.4, -0.6], [-0.4, -1.5],
                   [1.3, 0.3], [1.0, 1.4], [0.6, -1.25], [-0.8, 1.6]])
labels = np.array([1, 0, 0, 0, 1, 1, 1, 0])
start_weights = np.array([0.24, -0.80, 1.00, -0.50])

# ReLU keeps positive inputs and replaces negative inputs with zero.
x1 = points[:, 0]
x2 = points[:, 1]
hidden = np.maximum(0, np.column_stack([x1, -x1, x2, -x2]))

### Supplied figure setup

Run this cell once. It draws a boundary from four weights. The dashed line is the starting boundary. The solid line is the boundary after one update. Rings mark the examples used for that update.

In [ ]:
def draw_boundary(weights, batch_size):
    # Score a grid of positions so we can draw the line where A and B tie.
    axis = np.linspace(-2, 2, 150)
    x, y = np.meshgrid(axis, axis)
    features = np.maximum(0, np.stack([x, -x, y, -y], axis=-1))
    before = features @ start_weights - 0.30
    after = features @ weights - 0.30

    plt.figure(figsize=(4.6, 3.6))
    plt.contour(x, y, before, levels=[0], colors="gray", linestyles="--")
    plt.contour(x, y, after, levels=[0], colors="black")
    plt.scatter(x1[labels == 0], x2[labels == 0], label="Class A")
    plt.scatter(x1[labels == 1], x2[labels == 1], marker="^", label="Class B")
    plt.scatter(x1[:batch_size], x2[:batch_size], s=155, facecolors="none", edgecolors="black")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(f"One update: batch of {batch_size}")
    plt.axis("equal")
    plt.legend()
    plt.show()

### 1. Take one weight update

Run the baseline at learning rate 0.8 and batch size 8. Then try **1.6 and 8** to isolate learning rate. Finally try **0.8 and 1** to isolate batch size. Record the first weight each time. Each run starts from 0.24 for that weight.

In [ ]:
learning_rate = 0.8  # Edit one setting at a time.
batch_size = 8         # Choose 1 through 8 points.

# Take the first batch_size rows and compute their probabilities of class B.
batch_hidden = hidden[:batch_size]
score = batch_hidden @ start_weights - 0.30
probability = 1 / (1 + np.exp(-score))

# For cross-entropy, each example contributes (probability - label) times
# its hidden activation to each weight gradient. Average across the batch.
error = probability - labels[:batch_size]
gradient = batch_hidden.T @ error / batch_size

# Move against the gradient. Always begin with the same four weights.
new_weights = start_weights - learning_rate * gradient
print("Starting weights:", start_weights)
print("Updated weights: ", np.round(new_weights, 3))
draw_boundary(new_weights, batch_size)

Which comparison doubles the first weight change? Why does the one-point batch move the boundary differently?

**Your answer:** TODO

### 2. Which hidden units does dropout turn off?

Keep the lecture input (1, 1.4) and the original weights. Run at dropout rate 0.50, then at 0.00. The same random draws are used for both experiments. In the figure, **off** means dropout masked the unit. A displayed **0.0** means the unit was available but its ReLU output was zero.

In [ ]:
dropout_rate = 0.50  # Try values from 0 up to, but excluding, 1.
assert 0 <= dropout_rate < 1

# Positive parts of 1, -1, 1.4 and -1.4 for the fixed input.
activation = np.array([1.0, 0.0, 1.4, 0.0])
rng = np.random.default_rng(142)
keep = rng.random((2, 4)) >= dropout_rate  # Two passes, four units per pass.

# Training scales surviving units by 1 / (1 - rate). Inference uses the
# original activations. All three predictions use exactly the same weights.
training = activation * keep / (1 - dropout_rate)
outputs = np.vstack([training, activation])
scores = outputs @ start_weights - 0.30
prob_B = 1 / (1 + np.exp(-scores))

# Each row shows a complete pass. Read 'off' for masked units.
available = np.vstack([keep, np.ones(4, dtype=bool)])
plt.figure(figsize=(6.4, 2.6))
plt.imshow(available, cmap="summer", vmin=0, vmax=1, aspect="auto")
for row in range(3):
    for unit in range(4):
        label = f"{outputs[row, unit]:.1f}" if available[row, unit] else "off"
        plt.text(unit, row, label, ha="center", va="center")
plt.xticks(range(4), ["h1", "h2", "h3", "h4"])
plt.yticks(range(3), [f"Training A   P(B)={prob_B[0]:.3f}",
                      f"Training B   P(B)={prob_B[1]:.3f}",
                      f"Inference   P(B)={prob_B[2]:.3f}"])
plt.title(f"Hidden activations at dropout rate {dropout_rate:.2f}")
plt.tight_layout()
plt.show()

At rate 0.50, which active unit survives each training pass? What changes at rate 0.00?

**Your answer:** TODO

## Part B. Choose a model for MIMII machine sounds

Predict fan, pump, slider or valve from one second of sound. This course task uses only normal recordings from MIMII. The original benchmark studies anomalous sounds. The course split keeps all ten segments of a recording together. Machine IDs recur across partitions, so the validation results describe new recordings from familiar IDs.

### Supplied data setup

Run the next two cells unchanged. The first finds the course files in a local checkout or downloads them in Colab. The second assigns segments to their parent recording's partition and scales both partitions using training statistics.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import pandas as pd
import tensorflow as tf

folder = Path("mimii_logmel800")
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "data/mimii_logmel800"
    if (candidate / "logmel.npz").is_file():
        folder = candidate
        break

# Download missing files and verify each against the course release.
folder.mkdir(exist_ok=True)
base = "https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/mimii_logmel800/"
files = [("logmel.npz", "c6e1763b28155d7d08580176d53572953ed9f23475b5a199310087898daecd3a"),
         ("recordings.csv", "17f6261df25d44c5c99db761c7e8f632dd247281d9edbe391c2c6533a017341b")]
for name, expected_hash in files:
    path = folder / name
    if not path.exists():
        temporary = path.with_suffix(".download")
        urlretrieve(base + name, temporary)
        assert hashlib.sha256(temporary.read_bytes()).hexdigest() == expected_hash
        temporary.replace(path)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash

In [ ]:
records = pd.read_csv(folder / "recordings.csv")
with np.load(folder / "logmel.npz") as data:
    # Match each segment to its recording before choosing training or validation.
    assert np.array_equal(records["recording_id"], data["recording_id"])
    parent_row = data["recording_index"]
    partition = records["partition"].to_numpy()[parent_row]
    train = partition == "train"
    valid = partition == "validation"
    features = data["X"]
    sound_labels = data["y"]
    X_train = features[train]
    X_valid = features[valid]
    y_train = sound_labels[train]
    y_valid = sound_labels[valid]

# Use only training data to estimate the mean and standard deviation.
mean = X_train.mean()
sd = X_train.std()
X_train = (X_train - mean) / sd
X_valid = (X_valid - mean) / sd
print("Training segments:", len(y_train), " Validation segments:", len(y_valid))

### 3. Fit every width and learning-rate pair

Grid search tries every combination. Two widths and two rates give four fits. Compare them using **validation cross-entropy (CE)**. For one segment, CE is minus the natural log of the probability assigned to the true class. Lower is better.

Early stopping checks validation CE after each epoch. If it fails to improve for three epochs, training stops. The model restores the weights from its lowest validation CE. The 12-epoch cap still applies.

In [ ]:
# The two loops enumerate all four combinations. Each gets a fresh model.
results = []
lowest_loss = np.inf
for width in [16, 32]:
    for rate in [0.0003, 0.001]:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(9)
        model = tf.keras.Sequential([
            tf.keras.Input(shape=(32, 61)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(width, activation="relu"),
            tf.keras.layers.Dense(4, activation="softmax")
        ])
        optimizer = tf.keras.optimizers.Adam(learning_rate=rate)
        model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy")

        # Hold the split, batch size and stopping rule fixed across trials.
        stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                restore_best_weights=True)
        fit = model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                        epochs=12, batch_size=32, callbacks=[stop], verbose=0)

        # Python counts from zero. Add one to report the epoch as shown in class.
        losses = fit.history["val_loss"]
        best_epoch = np.argmin(losses) + 1
        best_loss = min(losses)
        results.append([width, rate, best_epoch, len(losses), best_loss])
        print("Finished width", width, "at learning rate", rate)

        # Save the best trial's curves for the next figure.
        if best_loss < lowest_loss:
            lowest_loss = best_loss
            best_history = fit.history

table = pd.DataFrame(results, columns=["Width", "Learning rate", "Best epoch",
                                       "Epochs run", "Validation CE"])
display(table)

### 4. Read the winning trial's learning curves

Find the lowest CE in the table. Compare its best epoch with its last epoch. If the minimum occurs at the cap, this run gives no evidence that validation loss had already started to rise.

In [ ]:
# Mark the epoch you would keep for this width and learning rate.
validation_loss = best_history["val_loss"]
epochs = np.arange(1, len(validation_loss) + 1)
chosen_epoch = np.argmin(validation_loss) + 1
plt.plot(epochs, best_history["loss"], label="Training")
plt.plot(epochs, validation_loss, label="Validation")
plt.scatter(chosen_epoch, min(validation_loss), color="black", label="Best checkpoint")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy per segment")
plt.legend()
plt.show()

Which width, rate and checkpoint would you keep? What does this split tell you about a new recording?

**Your answer:** TODO

**Optional challenge.** Try dropout rate 0.80 on the Part A input. Predict which pass will change before rerunning. Check whether inference still gives the same probability.

## Completion checklist

Edit the Markdown to change `[ ]` to `[x]`. Rendered checkboxes are not clickable.

- [ ] I compared weight updates at fixed batch size and at fixed learning rate.
- [ ] I traced a dropout mask from hidden activation to class probability.
- [ ] I selected a grid candidate and checkpoint using validation CE.

## Sources

Harsh Purohit and colleagues, [MIMII public 1.0](https://zenodo.org/records/3384388), CC BY-SA 4.0. The [course feature package](https://github.com/WSU-AI-in-ME/ai-in-me-1/tree/main/data/mimii_logmel800) provides channel-1 log-mel segments and the recording partitions.